# UAV — Video T4, khôi phục baseline chất lượng Grounding DINO

YOLO-World đã đạt tốc độ nhưng không đạt chất lượng trên video người dùng.
**Mặc định quay về Grounding DINO Base FP32, ảnh nguồn nguyên kích thước,
config/prompt giống pipeline ảnh. Đây chưa phải cấu hình đạt 9–15 FPS.**
Giữ giao diện gọn theo notebook sidewalk. Tracking giữa keyframe vẫn có thể sai;
chất lượng phải kiểm tra riêng. Backend World chỉ là ứng viên thử nghiệm và bị
chặn nếu audit thiếu, sai video/model hoặc không đạt. Agreement không phải nhãn chuẩn.
Để vừa đạt tốc độ vừa giữ chất lượng, cần fine-tune/huấn luyện detector nhẹ bằng
nhãn đã được người kiểm tra, không dùng zero-shot thay thế rồi chỉ xét FPS.


## 1. GPU và mã nguồn
Bật **Runtime → Change runtime type → T4 GPU**. Upload `UAV_lan_chiem_realtime_v3.zip`
vào `/content` hoặc `Drive/MyDrive/UAV`. Không dùng ZIP v1/v2 hoặc checkpoint cũ.


In [ ]:
from pathlib import Path
import hashlib, html, json, os, shutil, subprocess, sys, time, zipfile
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
DETECTOR_BACKEND = 'grounding-dino'  # Baseline đúng; world là ứng viên chưa được chấp nhận.
import torch
assert torch.cuda.is_available(), 'Hãy chọn T4 GPU runtime.'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)
from google.colab import drive, files
drive.mount('/content/drive')
ZIP_PATH = ''
candidates = [Path('/content/UAV_lan_chiem_realtime_v3.zip'),
              Path('/content/drive/MyDrive/UAV/UAV_lan_chiem_realtime_v3.zip')]
available = [p for p in candidates if p.is_file()]
if ZIP_PATH:
    package = Path(ZIP_PATH)
elif available:
    if len({hashlib.sha256(p.read_bytes()).hexdigest() for p in available}) > 1:
        raise RuntimeError('Có nhiều ZIP khác nhau; hãy điền ZIP_PATH.')
    package = available[0]
else:
    uploaded = files.upload()
    names = [n for n in uploaded if n.endswith('.zip')]
    if len(names) != 1: raise ValueError('Chọn đúng một ZIP v3.')
    package = Path(names[0]).resolve()
PROJECT = Path('/content/uav_lan_chiem_v3').resolve()
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(package) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)): raise ValueError('ZIP có tên trùng.')
    manifest = json.loads(archive.read('build_manifest.json'))
    if manifest.get('build_version') != 'video-v3-quality-gated':
        raise RuntimeError('Sai build; cần ZIP realtime v3 mới.')
    for member in archive.infolist():
        if not (PROJECT/member.filename).resolve().is_relative_to(PROJECT):
            raise ValueError('Unsafe ZIP path')
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('ZIP symlink not allowed')
    for name,digest in manifest['sha256'].items():
        if hashlib.sha256(archive.read(name)).hexdigest() != digest:
            raise ValueError('Sai checksum: '+name)
    archive.extractall(PROJECT)
sys.path[:] = [str(PROJECT)] + [entry for entry in sys.path if entry != str(PROJECT)]
for module_name in list(sys.modules):
    if module_name == 'lan_chiem' or module_name.startswith('lan_chiem.'):
        del sys.modules[module_name]
# Load the verified file directly: Colab may retain an older lan_chiem
# package/function in sys.modules after changing ZIPs. Do not reuse that cache.
import types, inspect
stream_path = PROJECT/'lan_chiem/colab_stream.py'
stream_bytes = stream_path.read_bytes()
stream_digest = hashlib.sha256(stream_bytes).hexdigest()
if stream_digest != manifest['sha256']['lan_chiem/colab_stream.py']:
    raise RuntimeError('Helper sau giải nén không khớp ZIP.')
stream_module = types.ModuleType('_uav_verified_stream_'+stream_digest[:12])
stream_module.__file__ = str(stream_path)
exec(compile(stream_bytes, str(stream_path), 'exec'), stream_module.__dict__)
run_streaming = stream_module.run_streaming
read_log_tail = stream_module.read_log_tail
if 'echo' not in inspect.signature(run_streaming).parameters:
    raise RuntimeError('ZIP helper quá cũ: thiếu echo. Upload gói v3 mới.')
print('Helper:', stream_path, '| signature:', inspect.signature(run_streaming))
print('Build:',manifest['build_version'],'| ZIP:',package)


In [ ]:
# Cài đặt là setup time, không tính vào FPS xử lý video.
setup_log = Path('/content/uav_lan_chiem_setup.log')
try:
    run_streaming([sys.executable,'-m','pip','install','-q','-r',
                   str(PROJECT/('lan_chiem/requirements-realtime.txt' if DETECTOR_BACKEND=='world' else 'lan_chiem/requirements-video.txt'))],
                  cwd=PROJECT,log_path=setup_log,echo=False)
    run_streaming([sys.executable,'-m','unittest',
                   'lan_chiem.tests.test_video','lan_chiem.tests.test_feature_cache',
                   'lan_chiem.tests.test_colab_stream','lan_chiem.tests.test_realtime','-q'],
                  cwd=PROJECT,log_path=setup_log,echo=False)
except Exception:
    print(read_log_tail(setup_log))
    raise
print('Dependencies và các test video đã đạt. Log chi tiết:',setup_log)

from IPython.display import display, HTML
status_handle = None
runtime_line = ''
def compact_status(line):
    global status_handle, runtime_line
    if line.startswith('[runtime]'): runtime_line=line
    if not line.startswith(('[prepare]','[load]','[runtime]','[warmup]','[video]','[audit]','[saved]')):
        return
    body = '<div style="padding:12px;border:1px solid #aaa;border-radius:8px">'
    if runtime_line: body += '<small>'+html.escape(runtime_line)+'</small><br>'
    if line.startswith('[video]'):
        import re
        fields=dict(re.findall(r'(\w+)=([^ ]+)',line))
        position=fields.get('frame','0/0').split('/')
        value=int(position[0])+1
        total=int(position[1]) if position[1].isdigit() else max(value,1)
        body += f'<progress value="{value}" max="{total}" style="width:100%"></progress>'
        text=f"Frame {value}/{total} | FPS trung bình: {fields.get('avg_fps','?')} | AI/frame: {fields.get('AI','?')} | Vật đang theo dõi: {fields.get('tracks','?')} | Còn: {fields.get('ETA','?')}"
        body += '<b>'+html.escape(text)+'</b>'
    else:
        body += '<b>'+html.escape(line)+'</b>'
    body += '</div>'
    if status_handle is None: status_handle=display(HTML(body),display_id=True)
    else: status_handle.update(HTML(body))


## 2. Chọn video và cấu hình
Giữ toàn bộ frame. `MAX_FRAMES=0` chạy hết video; có thể đặt 120 để đo thử trước.
Ảnh/video làm việc giới hạn 960 px; JSON/CSV vẫn dùng tọa độ ảnh nguồn.
Chọn video từ Drive hoặc để trống INPUT_VIDEO và upload.


In [ ]:
INPUT_VIDEO = '/content/drive/MyDrive/UAV/123.mp4'
MODEL_WEIGHTS = 'yolov8m-worldv2.pt'
IMAGE_SIZE = 960
MAX_FRAMES = 60
RUN_AUDIT = True    # Đối chiếu 4 frame với DINO; chi phí này đo riêng.
from uuid import uuid4
RUN_ID = DETECTOR_BACKEND.replace('-','_') + '_quality_' + uuid4().hex[:8]
# Experimental: same DINO model/prompts/native image size; validate quality and T4 speed first.
DINO_BATCH_EXPERIMENT = True
VIDEO_CONFIG = ('lan_chiem/video_dino_batch.yaml' if DINO_BATCH_EXPERIMENT else 'lan_chiem/video_quality.yaml') if DETECTOR_BACKEND=='grounding-dino' else 'lan_chiem/video_realtime.yaml'
OUTPUT = Path('/content/uav_object_results')/RUN_ID
AUDIT_OUTPUT = OUTPUT.with_name(RUN_ID+'_audit')

if INPUT_VIDEO:
    video_source = Path(INPUT_VIDEO)
else:
    uploaded=files.upload()
    names=[n for n in uploaded if Path(n).suffix.lower() in {'.mp4','.avi','.mov','.mkv','.webm'}]
    if len(names)!=1: raise ValueError('Chọn đúng một video.')
    video_source=Path(names[0]).resolve()
if not video_source.is_file(): raise FileNotFoundError(video_source)
LOCAL_VIDEO=Path('/content/uav_inputs')/video_source.name
LOCAL_VIDEO.parent.mkdir(parents=True,exist_ok=True)
if LOCAL_VIDEO.resolve()!=video_source.resolve(): shutil.copy2(video_source,LOCAL_VIDEO)
import cv2
cap=cv2.VideoCapture(str(LOCAL_VIDEO))
if not cap.isOpened(): raise RuntimeError('Không đọc được video')
count=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps=cap.get(cv2.CAP_PROP_FPS)
print(f'Video: {count} frames | {fps:.2f} FPS | {count/max(fps,1e-6):.2f}s')
print('Source:',int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), 'x',int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
cap.release()
print('Output:',OUTPUT)


## 3. Đối chiếu chất lượng detector mới (khuyến nghị)
Audit chạy Grounding DINO FP32 trên 4 frame trải đều clip rồi so YOLO-World FP16.
Ảnh `_reference.jpg` và `_candidate.jpg` giúp kiểm tra vật nhỏ/lớp khó.
**Agreement không chứng minh đúng ngoài thực tế**; lớp không có trong mẫu chưa được kiểm định.
Nếu screen không đạt, kết quả realtime chỉ là thử nghiệm tốc độ, chưa nên đưa vào sử dụng.

Audit chỉ chạy khi DETECTOR_BACKEND=world. Audit không đạt sẽ chặn cell phân tích tiếp theo.


In [ ]:
audit_report=None
if DETECTOR_BACKEND == 'world' and RUN_AUDIT:
    audit_log=AUDIT_OUTPUT.with_suffix('.log')
    t=time.perf_counter()
    try:
        run_streaming([sys.executable,'-u','-m','lan_chiem.video_audit',
                       '--input',str(LOCAL_VIDEO),'--output',str(AUDIT_OUTPUT),
                       '--weights',MODEL_WEIGHTS,'--imgsz',str(IMAGE_SIZE),
                       '--work-max-side','960','--samples','4','--device','cuda'],
                      cwd=PROJECT,log_path=audit_log,on_line=compact_status,echo=False)
    except Exception:
        print(read_log_tail(audit_log))
        raise
    audit_report=json.loads((AUDIT_OUTPUT/'audit.json').read_text())
    print('Audit time (riêng, không tính FPS):',round(time.perf_counter()-t,1),'s')
    print('DINO reference retention:',audit_report['reference_retention'])
    print('Candidate agreement:',audit_report['candidate_agreement'])
    print('Agreement screen:', 'PASS' if audit_report['agreement_screen_passed'] else 'NOT PASS')
    print('Lớp chưa có mẫu reference:',audit_report['classes_without_reference'])
    print('Ảnh đối chiếu:',AUDIT_OUTPUT,'| quality_verified=False')
else:
    print('Baseline DINO: không thay model. Nếu chọn World thì audit đạt là điều kiện bắt buộc.')

if audit_report is not None:
    # Ảnh kiểm tra nằm trong vùng thu gọn, không làm tràn log.
    import base64
    panels=[]
    for reference in sorted(AUDIT_OUTPUT.glob('*_reference.jpg')):
        candidate=reference.with_name(reference.name.replace('_reference','_candidate'))
        if candidate.exists():
            pictures=''.join('<img width="48%" src="data:image/jpeg;base64,'+base64.b64encode(path.read_bytes()).decode()+'">' for path in (reference,candidate))
            panels.append('<p>'+reference.stem.split('_')[0]+': DINO (trái) / World (phải)</p>'+pictures)
    if panels: display(HTML('<details><summary>Mở ảnh đối chiếu chất lượng</summary>'+''.join(panels)+'</details>'))


## 4. Phân tích baseline chất lượng

Một ô tiến độ, log riêng. Mặc định Grounding DINO FP32, không giảm ảnh làm việc,
không bỏ frame. MAX_FRAMES=60 để kiểm tra chất lượng trước, 0 để chạy hết.
Bảng FPS vẫn báo đúng, không tuyên bố realtime khi chưa đạt.


In [ ]:
# A failed candidate cannot proceed even if this cell is run manually.
from lan_chiem.quality_gate import require_candidate_audit
from lan_chiem.video_io import file_hash
require_candidate_audit(DETECTOR_BACKEND, audit_report, file_hash(LOCAL_VIDEO), MODEL_WEIGHTS, IMAGE_SIZE, 960)
RUN_LOG=OUTPUT.with_suffix('.log')
command=[sys.executable,'-u','-m','lan_chiem.video_predict',
         '--input',str(LOCAL_VIDEO),'--output',str(OUTPUT),
         '--device','cuda','--backend',DETECTOR_BACKEND,'--weights',MODEL_WEIGHTS,
         '--imgsz',str(IMAGE_SIZE),'--video-config',VIDEO_CONFIG,
         '--max-frames',str(MAX_FRAMES)]
run_started=time.perf_counter()
try:
    run_streaming(command,cwd=PROJECT,log_path=RUN_LOG,on_line=compact_status,echo=False)
except BaseException:
    print(read_log_tail(RUN_LOG))
    raise
cli_wall=time.perf_counter()-run_started
shutil.copy2(RUN_LOG,OUTPUT/'run.log')
profile=json.loads((OUTPUT/'summary.json').read_text())
cli_fps=profile['processed_frames']/max(cli_wall,1e-6)
profile.update(cli_wall_seconds=cli_wall,cli_processed_fps=cli_fps,cli_target_met=cli_fps>=9 and not profile['frame_skipping_enabled'])
(OUTPUT/'summary.json').write_text(json.dumps(profile,ensure_ascii=False,indent=2),encoding='utf-8')
rows=[('Frames',f"{profile['processed_frames']}/{count}"),
      ('Xử lý hết mọi frame',str(profile['all_source_frames_processed'])),
      ('CLI wall kể cả Python startup',f'{cli_wall:.2f} s'),
      ('FPS toàn CLI kể cả Python startup',f'{cli_fps:.2f}'),
      ('FPS kể cả load/warm-up/final export',f"{profile['processed_fps']:.2f}"),
      ('FPS pipeline sau load/warm-up',f"{profile['steady_pipeline_fps_this_run']:.2f}"),
      ('Mục tiêu tối thiểu 9 FPS', 'ĐẠT' if profile['cli_target_met'] else 'CHƯA ĐẠT'),
      ('Mục tiêu ưu tiên 15 FPS','ĐẠT' if cli_fps>=15 else 'CHƯA ĐẠT'),
      ('Cold end-to-end >=9 FPS',str(profile['end_to_end_target_met'])),
      ('Tốc độ so với nguồn (>=1 mới theo kịp nguồn)',f"{profile['speed_vs_source']:.2f}x"),
      ('P95 frame latency',f"{profile['latency_p95_seconds_this_run']:.3f} s"),
      ('Model / warm-up',f"{profile['stage_seconds']['model_load']:.2f} / {profile['warmup_seconds_this_run']:.2f} s"),
      ('AI / tracking-only frames',f"{profile['counts'].get('detector_frames',0)} / {profile['counts'].get('tracking_only_frames',0)}"),
      ('Độ chính xác nhãn chuẩn','CHƯA KIỂM ĐỊNH')]
body='<table>'+''.join('<tr><td style="padding:6px">'+html.escape(k)+'</td><td><b>'+html.escape(v)+'</b></td></tr>' for k,v in rows)+'</table>'
display(HTML(body))
print('Stages:',profile['stage_seconds'])
if not profile['cli_target_met']:
    print('CHƯA ĐẠT 9 FPS: gửi summary.json và run.log; không lấy FPS playback hoặc frame TRACK để thay thế.')
if audit_report is not None and not audit_report['agreement_screen_passed']:
    print('Chất lượng lệch DINO: cần kiểm tra ảnh audit/nhãn chuẩn hoặc fine-tune trước khi sử dụng.')
print('Profile:',OUTPUT/'summary.json')


## 5. Xem video và sao lưu
Chuyển H.264 để xem trên trình duyệt được đo riêng, không cộng vào FPS pipeline.
Video không có âm thanh. Không đổi FPS phát video. Số track chưa phải số vật duy nhất.


In [ ]:
from IPython.display import Video
raw_result=OUTPUT/'annotated.mp4'
web_result=OUTPUT/'preview_web.mp4'
t=time.perf_counter()
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(raw_result),
                '-c:v','libx264','-preset','veryfast','-crf','22','-pix_fmt','yuv420p',str(web_result)],check=True)
print('Browser conversion:',round(time.perf_counter()-t,2),'s (riêng)')
display(Video(str(web_result),embed=True,width=960))


In [ ]:
# Sao lưu sau xử lý để Drive I/O không làm nhiễu benchmark.
BACKUP=Path('/content/drive/MyDrive/UAV/video_results')/RUN_ID
t=time.perf_counter()
shutil.copytree(OUTPUT,BACKUP,dirs_exist_ok=True)
if AUDIT_OUTPUT.exists():shutil.copytree(AUDIT_OUTPUT,BACKUP/'audit',dirs_exist_ok=True)
print('Drive backup:',BACKUP,'| time:',round(time.perf_counter()-t,2),'s (riêng)')
archive=shutil.make_archive('/content/'+RUN_ID+'_results','zip',OUTPUT)
files.download(archive)
